# Colab 실행 안내

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/glasslego/2026-ml-study/blob/main/notebooks/DLFS_code/03_dlfs/Code_tensorflow.ipynb)

이 노트북은 같은 디렉토리의 `Code.ipynb` (밑바닥부터 numpy 로 구현한 신경망) 와
**같은 추상화/같은 흐름**으로 동일한 회귀 문제를 **TensorFlow / Keras** 로 다시 푼다.

흐름:

1. 환경 준비 (Colab/Local 공통)
2. Operation 대응 — `tf.Variable` + `tf.GradientTape`
3. Layer 대응 — `tf.keras.layers.Dense`
4. Loss 대응 — `tf.keras.losses.MeanSquaredError`
5. NeuralNetwork 대응 — `tf.keras.Sequential`
6. Optimizer 대응 — `tf.keras.optimizers.SGD`
7. Trainer 대응 — (저수준) `tf.GradientTape` 학습 루프, (고수준) `model.fit()`
8. 데이터 로드 / 표준화 / 분할 (Boston deprecated → California Housing)
9. 3가지 모델 비교 (선형회귀 / 1은닉층 / 2은닉층)

> 같은 연산을 책의 numpy 코드, PyTorch (`Code_pytorch.ipynb`), TensorFlow 셋이 어떻게
> 표현하는지 비교하면서 보면 "무엇을 자동으로 해 주는가" 가 명확해진다.


In [ ]:
# 공통 실행 환경 준비
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/glasslego/2026-ml-study.git'
REPO_NAME = '2026-ml-study'
DLFS_SUBDIR = 'notebooks/DLFS_code'

if 'google.colab' in sys.modules:
    _bootstrap_clone_root = Path('/content') / REPO_NAME
    if not _bootstrap_clone_root.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(_bootstrap_clone_root)], check=True)
    _bootstrap_repo_root = _bootstrap_clone_root / DLFS_SUBDIR
else:
    _bootstrap_repo_root = None
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / 'README.md').exists() and (candidate / 'lincoln').exists():
            _bootstrap_repo_root = candidate
            break
    if _bootstrap_repo_root is None:
        raise FileNotFoundError('DLFS_code 저장소 루트를 찾지 못했습니다.')

if str(_bootstrap_repo_root) not in sys.path:
    sys.path.insert(0, str(_bootstrap_repo_root))

from notebook_env import prepare_notebook_environment

REPO_ROOT = prepare_notebook_environment(
    notebook_dir='03_dlfs',
    needs_lincoln=False,
    ensure_mnist=False,
)


In [ ]:
# Colab 에는 TF 가 기본 설치되어 있지만, 로컬에 없을 수도 있어 안전하게 보강한다.
try:
    import tensorflow as tf  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tensorflow'], check=True)
    import tensorflow as tf  # noqa: F401


In [ ]:
# 핵심 import.
# - tf.Variable        : 학습 가능한 파라미터
# - tf.GradientTape    : 자동미분 컨텍스트 (책의 backward 식들을 대체)
# - tf.keras.layers    : Layer 모듈 (책의 Layer 추상화에 대응)
# - tf.keras.losses    : Loss
# - tf.keras.optimizers: Optimizer
import numpy as np
import tensorflow as tf
from tensorflow import keras
from typing import List

# 재현성: 책의 seed=20190501 과 동일한 의도.
SEED = 20190501
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

print(f"tensorflow version: {tf.__version__}")


# 1. Operation 대응 — `tf.Variable` + `tf.GradientTape`

책의 `Operation` 추상화는 forward 와 backward 를 직접 구현해야 했다.
TensorFlow 에서는 forward 식을 `tf.GradientTape` 안에서 실행하면, **테이프가 모든 연산을 기록**해
`tape.gradient(loss, [vars])` 한 줄로 chain rule 결과를 얻을 수 있다.


In [ ]:
# numpy 와 TensorFlow 의 forward/backward 비교 (작은 예시)
rng = np.random.default_rng(0)
X_np = rng.standard_normal((4, 3)).astype(np.float32)
W_np = rng.standard_normal((3, 2)).astype(np.float32)
b_np = rng.standard_normal((1, 2)).astype(np.float32)


# (a) 책의 방식
def forward_numpy(X, W, b):
    Y = X @ W + b
    S = 1.0 / (1.0 + np.exp(-Y))
    L = float(S.sum())
    return Y, S, L


_, _, L_np = forward_numpy(X_np, W_np, b_np)
print("numpy loss:", L_np)

# (b) TensorFlow — tape.gradient 가 자동으로 chain rule 처리.
X = tf.constant(X_np)
W = tf.Variable(W_np)
b = tf.Variable(b_np)

with tf.GradientTape() as tape:
    Y = tf.matmul(X, W) + b
    S = tf.sigmoid(Y)
    L = tf.reduce_sum(S)

dL_dW, dL_db = tape.gradient(L, [W, b])
print("tf loss:", L.numpy())
print("dL/dW shape:", dL_dW.shape, "  dL/db shape:", dL_db.shape)


# 2. Layer 대응 — `tf.keras.layers.Dense`

책의 `Dense` 층은 `[WeightMultiply -> BiasAdd -> activation]` 세 Operation 의 합성이었다.
Keras 의 `Dense(units, activation=...)` 한 줄이 같은 일을 한다.
가중치는 첫 호출에서 입력 차원이 결정되는 시점에 자동으로 만들어진다 (책의 "지연 초기화" 와 같은 방식).


In [ ]:
# 작은 예시: Dense 레이어가 forward / backward / 파라미터 관리를 모두 캡슐화한다.
dense = keras.layers.Dense(units=2, activation='sigmoid')
y = dense(tf.random.normal((4, 3)))   # 첫 호출에서 input_dim 을 보고 가중치 (3, 2) 와 편향 (2,) 생성
print("output shape:", y.shape)
print("trainable params:", [v.name + ' ' + str(v.shape) for v in dense.trainable_variables])


# 3. Loss 대응 — `MeanSquaredError`

책의 `MeanSquaredError` 와 똑같이 `(1/N) sum (y_hat - y)^2` 를 계산한다.
역전파 식은 GradientTape 가 자동으로 만들어 준다.


In [ ]:
mse = keras.losses.MeanSquaredError()

y_true = tf.constant([[1.0], [1.0], [1.0]])
y_hat  = tf.Variable([[1.0], [2.0], [3.0]])

with tf.GradientTape() as tape:
    loss = mse(y_true, y_hat)
grad = tape.gradient(loss, y_hat)
print("loss:", loss.numpy())                 # mean((0, 1, 4)) = 5/3
print("dL/dy_hat:", tf.squeeze(grad).numpy())  # 2*(y_hat - y)/N = [0, 2/3, 4/3]


# 4. NeuralNetwork 대응 — `keras.Sequential`

책의 `NeuralNetwork(layers=[Dense(...), ...])` 와 같은 의미.


In [ ]:
def make_network(layer_sizes: List[int], activations: List[str]) -> keras.Model:
    """책의 NeuralNetwork(layers=[Dense(...), ...]) 와 같은 역할.

    Args:
        layer_sizes: [hidden1, ..., 출력차원] (입력 차원은 첫 forward 에서 결정됨)
        activations: 각 Dense 층의 활성화 (length = len(layer_sizes))
    """
    assert len(activations) == len(layer_sizes), \
        "activation 개수는 layer_sizes 와 같아야 함"

    model = keras.Sequential()
    for units, act in zip(layer_sizes, activations):
        # activation=None 은 책의 Linear (항등) 와 동일.
        model.add(keras.layers.Dense(units=units, activation=act))
    return model


# 5. Optimizer 대응 — `keras.optimizers.SGD`

책의 `SGD.step()` 은 `param -= lr * grad` 였다.
Keras 의 `keras.optimizers.SGD(learning_rate=0.01)` 가 같은 일을 한다.
PyTorch 와 달리 `optimizer.apply_gradients(zip(grads, vars))` 형태로 사용한다.


# 6. Trainer 대응 — 저수준 GradientTape 학습 루프

책의 `Trainer.fit()` 과 가장 가까운 형태.
직접 epoch 루프를 돌리며 `tape.gradient → optimizer.apply_gradients` 를 호출한다.
이 형태가 책의 `train_batch + optim.step` 과 1:1 로 매칭된다.


In [ ]:
import copy

def shuffle_indices(n: int, rng: np.random.Generator) -> np.ndarray:
    """책의 permute_data 와 같은 의미. 매 epoch 시작에 호출."""
    return rng.permutation(n)


class TFTrainer:
    """책의 Trainer 를 TensorFlow GradientTape 위에 그대로 옮긴 미니 트레이너.

    같은 부분: eval_every 마다 검증 손실 측정, best 갱신, 나빠지면 백업으로 복원하고 종료.
    TF 가 대신해 주는 부분: chain rule (tape.gradient), 파라미터 갱신 (optimizer.apply_gradients).
    """

    def __init__(self, net: keras.Model, optimizer: keras.optimizers.Optimizer, loss_fn):
        self.net = net
        self.optimizer = optimizer
        self.loss_fn = loss_fn
        self.best_loss = float('inf')

    def _train_step(self, X_batch, y_batch):
        # 책의 train_batch 와 1:1 대응.
        with tf.GradientTape() as tape:
            preds = self.net(X_batch, training=True)
            loss = self.loss_fn(y_batch, preds)
        grads = tape.gradient(loss, self.net.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.net.trainable_variables))
        return loss

    def fit(self,
            X_train: np.ndarray, y_train: np.ndarray,
            X_test: np.ndarray, y_test: np.ndarray,
            epochs: int = 50,
            eval_every: int = 10,
            batch_size: int = 32,
            seed: int = SEED) -> None:

        rng = np.random.default_rng(seed)
        N = X_train.shape[0]
        self.best_loss = float('inf')

        # 가중치 미리 build: 첫 forward 가 일어나야 trainable_variables 가 만들어진다.
        # (그래야 deepcopy 로 백업/복원이 가능)
        _ = self.net(X_train[:1])

        for e in range(epochs):
            # eval 직전 모델 가중치 백업 (early stopping 시 복원용)
            if (e + 1) % eval_every == 0:
                last_weights = [v.numpy().copy() for v in self.net.trainable_variables]

            # --- 한 epoch 학습 ---
            perm = shuffle_indices(N, rng)        # 책의 permute_data 에 대응
            X_perm, y_perm = X_train[perm], y_train[perm]
            for start in range(0, N, batch_size):
                X_batch = X_perm[start:start + batch_size]
                y_batch = y_perm[start:start + batch_size]
                self._train_step(X_batch, y_batch)

            # --- 검증 손실 + early stopping ---
            if (e + 1) % eval_every == 0:
                val_preds = self.net(X_test, training=False)
                val_loss = float(self.loss_fn(y_test, val_preds).numpy())

                if val_loss < self.best_loss:
                    print(f"{e + 1} 에폭에서 검증 데이터에 대한 손실값: {val_loss:.3f}")
                    self.best_loss = val_loss
                else:
                    print(
                        f"{e + 1} 에폭에서 손실값이 증가했다. 마지막으로 측정한 손실값은 "
                        f"{e + 1 - eval_every} 에폭까지 학습된 모델에서 계산된 {self.best_loss:.3f} 이다."
                    )
                    # 백업본으로 복원하고 학습 종료
                    for v, w in zip(self.net.trainable_variables, last_weights):
                        v.assign(w)
                    break


# 7. 평가 함수

책의 `mae`, `rmse`, `eval_regression_model` 과 같은 의미의 헬퍼.


In [ ]:
def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(y_true - y_pred)))


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def eval_regression_model(model: keras.Model, X_test: np.ndarray, y_test: np.ndarray) -> None:
    preds = model(X_test, training=False).numpy().reshape(-1, 1)
    print("평균절대오차: {:.2f}".format(mae(y_test, preds)))
    print()
    print("제곱근 평균제곱오차 {:.2f}".format(rmse(y_test, preds)))


# 8. 데이터 로드 — Boston 대신 California Housing

`sklearn.datasets.load_boston` 은 1.2 부터 제거되었다. 같은 형태(여러 수치 특성 → 연속 target) 인
California Housing 으로 대체한다. 절차는 책과 동일: 표준화 → train/test 분할 → (N, 1) reshape.


In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

cali = fetch_california_housing()
data = cali.data            # shape: (20640, 8)
target = cali.target        # shape: (20640,)
features = cali.feature_names

scaler = StandardScaler()
data = scaler.fit_transform(data).astype(np.float32)
target = target.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    data, target, test_size=0.3, random_state=80718
)

# 책과 동일하게 target 을 (N, 1) 로 만든다.
y_train = y_train.reshape(-1, 1)
y_test  = y_test.reshape(-1, 1)

print("X_train:", X_train.shape, " y_train:", y_train.shape)
print("X_test :", X_test.shape,  " y_test :", y_test.shape)
print("입력 차원:", X_train.shape[1])


# 9. 3가지 모델 비교 — 저수준 학습 루프

책과 동일하게 세 가지 모델을 같은 방식으로 학습/평가한다.

| 이름 | 구조                                                   |
|------|--------------------------------------------------------|
| lr   | `[Dense(1)]`                                           |
| nn   | `[Dense(13, sigmoid) → Dense(1)]`                      |
| dl   | `[Dense(13, sigmoid) → Dense(13, sigmoid) → Dense(1)]` |


In [ ]:
HIDDEN = 13

keras.utils.set_random_seed(SEED)
lr_model = make_network(layer_sizes=[1], activations=[None])

keras.utils.set_random_seed(SEED)
nn_model = make_network(layer_sizes=[HIDDEN, 1], activations=['sigmoid', None])

keras.utils.set_random_seed(SEED)
dl_model = make_network(layer_sizes=[HIDDEN, HIDDEN, 1], activations=['sigmoid', 'sigmoid', None])

# build 는 첫 forward 에서 자동으로 일어난다.
_ = lr_model(X_train[:1])
_ = nn_model(X_train[:1])
_ = dl_model(X_train[:1])
lr_model.summary()
nn_model.summary()
dl_model.summary()


In [ ]:
# 모델 1: 선형회귀
trainer = TFTrainer(lr_model, keras.optimizers.SGD(learning_rate=0.01), keras.losses.MeanSquaredError())
trainer.fit(X_train, y_train, X_test, y_test, epochs=50, eval_every=10, seed=SEED)
print()
eval_regression_model(lr_model, X_test, y_test)


In [ ]:
# 모델 2: 은닉층 1개 + sigmoid
trainer = TFTrainer(nn_model, keras.optimizers.SGD(learning_rate=0.01), keras.losses.MeanSquaredError())
trainer.fit(X_train, y_train, X_test, y_test, epochs=50, eval_every=10, seed=SEED)
print()
eval_regression_model(nn_model, X_test, y_test)


In [ ]:
# 모델 3: 은닉층 2개 + sigmoid ("deep")
trainer = TFTrainer(dl_model, keras.optimizers.SGD(learning_rate=0.01), keras.losses.MeanSquaredError())
trainer.fit(X_train, y_train, X_test, y_test, epochs=50, eval_every=10, seed=SEED)
print()
eval_regression_model(dl_model, X_test, y_test)


# 10. (보너스) 같은 모델을 고수준 `model.fit()` 으로

위의 `TFTrainer` 는 책과 1:1 매칭이 잘 보이도록 GradientTape 학습 루프를 직접 짠 것이다.
실무에서는 `model.compile() + model.fit()` 한 줄이면 같은 일을 더 간결히 할 수 있다.

같은 dl 구조를 다시 만들어 비교한다.


In [ ]:
keras.utils.set_random_seed(SEED)
dl_model_keras = make_network(layer_sizes=[HIDDEN, HIDDEN, 1], activations=['sigmoid', 'sigmoid', None])
dl_model_keras.compile(
    optimizer=keras.optimizers.SGD(learning_rate=0.01),
    loss=keras.losses.MeanSquaredError(),
    metrics=[keras.metrics.MeanAbsoluteError(name='mae')],
)
history = dl_model_keras.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=32,
    verbose=0,
)
print(f"마지막 epoch 검증 손실: {history.history['val_loss'][-1]:.3f}")
eval_regression_model(dl_model_keras, X_test, y_test)


# 정리: 책의 numpy 코드와 TensorFlow 의 매핑

| 책의 추상화 (numpy)              | TensorFlow / Keras 대응                       |
|----------------------------------|-----------------------------------------------|
| `Operation` / `_input_grad`      | `tf.Variable` + `tf.GradientTape.gradient`    |
| `WeightMultiply` + `BiasAdd`     | `keras.layers.Dense` (가중치+편향 통합)       |
| `Sigmoid`, `Linear`              | `activation='sigmoid'`, `activation=None`     |
| `Layer.forward / backward`       | `keras.layers.Layer.call` (+ tape backward)   |
| `Dense`                          | `keras.layers.Dense`                          |
| `Loss.forward / backward`        | `keras.losses.MeanSquaredError` + tape grad   |
| `NeuralNetwork`                  | `keras.Sequential` / 사용자 `keras.Model`     |
| `SGD.step`                       | `keras.optimizers.SGD.apply_gradients`        |
| `Trainer.generate_batches`       | 직접 슬라이싱 (또는 `tf.data.Dataset.batch`)  |
| `Trainer.fit` (early stop)       | `TFTrainer.fit` (직접) / `model.fit()` 고수준 |

GradientTape 한 줄이 책의 모든 `_input_grad`, `_param_grad` 구현을 대체한다는 것을 확인할 수 있다.
